In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_app = pd.read_csv('APP_Scaled_LAG.csv')

In [ ]:
df_app

In [ ]:
df_app.columns

In [ ]:
app_feature_list = ['phq1',
       'phq2', 'meal_morning', 'meal_lunch', 'meal_dinner', 'meal_night',
       'sleep_satisfy', 'sleep_disturb', 'alcohol_when_morning',
       'alcohol_when_lunch', 'alcohol_when_dinner', 'alcohol_when_night',
       'amount_beer_glass', 'amount_soju_glass', 'amount_wine_glass',
       'amount_whiskey_glass', 'amount_rice_glass', 'alcohol_amount_etc_True',
       'alcohol_why_stress_True', 'alcohol_why_social_True',
       'alcohol_why_happy_True', 'alcohol_why_sleep_True',
       'alcohol_why_lonely_True', 'alcohol_why_unavoidable_True',
       'alcohol_why_habit_True', 'alcohol_why_pairing_True',
       'alcohol_with_whom_other', 'alcohol_with_whom_with_colleagues',
       'alcohol_with_whom_with_family', 'alcohol_with_whom_with_friends',
       'alcohol_with_whom_with_partner', 'alcohol_how_long_less_than_2_hours',
       'alcohol_how_long_less_than_4_hours',
       'alcohol_how_long_more_than_4_hours', 'alcohol_how_long_other',
       'alcohol_where_etc', 'alcohol_where_friends_home', 'alcohol_where_home',
       'alcohol_where_outdoor', 'alcohol_where_pub',
       'alcohol_where_restaurant', 'alcohol_blackout_never',
       'alcohol_blackout_other', 'alcohol_blackout_partially',
       'sleep_minute_adjusted']

In [ ]:
len(app_feature_list)

In [ ]:
TEST_DAYS = 5
VAL_DAYS = 3

In [ ]:
df_app.groupby('user_order').size().unique()

In [ ]:
df_app_train_list = []

for user in df_app.user_order.unique():
    df_user = df_app[df_app.user_order == user]
    T = len(df_user)
    df_app_train_list.append(df_user.iloc[:T - 8])
    
    
df_app_train = pd.concat(df_app_train_list).reset_index(drop = True)

In [ ]:
df_app_train.groupby('user_id').size()

## Feature Selection

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

In [ ]:
df_app_train

In [ ]:
df_app_train.columns

In [ ]:
y = df_app_train.tmrw_alcohol_desire

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(df_app_train.drop(columns = ['user_order', 'user_id', 'day_order', 'date', 'alcohol_desire','tmrw_alcohol_desire'], axis = 1, errors = 'ignore'), y)


In [ ]:
selector = SelectFromModel(rf, max_features=16, prefit=True)

In [ ]:
rf.feature_importances_

In [ ]:
selector.get_support()

In [ ]:
selected_app_features = df_app_train.drop(columns = ['user_order', 'user_id', 'day_order', 'date', 'alcohol_desire','tmrw_alcohol_desire'], axis = 1, errors = 'ignore').columns[selector.get_support()]

In [ ]:
selected_app_features

In [ ]:
final_columns = ['user_id', 'user_order', 'date', 'day_order', 'date_diff','phq1', 'phq2', 'meal_morning', 'meal_lunch', 'meal_dinner',
       'meal_night', 'sleep_satisfy', 'sleep_disturb', 'amount_beer_glass',
       'amount_soju_glass', 'sleep_minute_adjusted','tmrw_alcohol_desire']

In [ ]:
df_final = df_app[final_columns]

In [ ]:
df_final

In [ ]:
df_final.to_csv('App_FeatureSelection.csv', index = False)